# Part 4: Multi-Query Retrieval — 질문 확장으로 검색 보완

이 섹션에서는:
1. **이론**: 단일 쿼리의 한계와 질문 확장의 필요성
2. **작동 방식**: LLM이 질문을 다양한 관점으로 변형하는 과정
3. **데모**: 확장된 쿼리들이 각각 어떤 문서를 가져오는지 시각화
4. **노이즈 리스크**: 확장 질문이 원질문과 어긋나는 경우

## 4-1. 단일 쿼리의 한계

사용자는 질문을 **한 가지 표현**으로만 작성합니다. 하지만 같은 의도라도 표현은 다양할 수 있습니다:

| 원래 질문 | 같은 의도의 다른 표현 |
|-----------|---------------------|
| "삼성전자 DS 부문 영업이익은?" | "삼성 반도체 사업부 수익은?" |
| "SK하이닉스 목표주가는?" | "SK하이닉스 투자의견 및 적정가는?" |
| "누가 더 실적이 좋은가?" | "영업이익 비교", "실적 우위 기업" |

문서에 "반도체 사업부"로 기술되어 있는데 사용자가 "DS 부문"으로 물으면,
의미 검색이라도 최적의 문서를 놓칠 수 있습니다.

**Multi-Query Retrieval**은 LLM을 사용해 원질문을 여러 관점으로 변형(확장)하고,
각 변형 질문으로 검색한 결과를 합쳐서 이 문제를 해결합니다.

## 4-2. Multi-Query Retrieval 작동 방식

```
사용자 질문
    │
    ├── LLM이 3가지 변형 질문 생성
    │     ├── 변형 1 → 검색 → 문서 A, B, C
    │     ├── 변형 2 → 검색 → 문서 B, D, E
    │     └── 변형 3 → 검색 → 문서 A, E, F
    │
    └── 원본 질문 → 검색 → 문서 A, B, G
                      │
                      ▼
              합집합: A, B, C, D, E, F, G
              (중복 제거)
```

**장점**: 단일 질문으로 놓치기 쉬운 문서를 변형 질문이 보완

**비용**: LLM 호출 1회 추가 (질문 확장용, 가벼운 모델 사용 가능)

In [8]:
# 패키지 임포트 및 데이터 로드
import pickle
import time
import logging
from langchain.retrievers import MultiQueryRetriever, EnsembleRetriever
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
load_dotenv()

# MultiQueryRetriever가 생성한 쿼리를 확인하기 위한 로깅 설정
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.DEBUG)

with open("splits_v2.pkl", "rb") as f:
    splits_data = pickle.load(f)
splits = splits_data["vlm"]

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
llm = ChatOpenAI(model="gpt-4o", temperature=0)
TOP_K = 5

RAG_PROMPT = """다음 맥락을 바탕으로 질문에 답하세요. 맥락에 없는 내용은 추측하지 마세요.

맥락:
{context}

질문: {question}
답변:"""

print(f"VLM 추출 청크 수: {len(splits)}")

VLM 추출 청크 수: 105


## 4-3. MultiQueryRetriever 구성

LangChain의 `MultiQueryRetriever`를 사용하면 질문 확장 → 각 질문별 검색 → 결과 합집합을 **자동으로** 처리합니다.

`from_llm()`으로 간단하게 생성할 수 있으며, 확장 질문 생성에는 가벼운 모델(`gpt-4o-mini`)을 사용합니다.

In [16]:
# 기본 Ensemble Retriever 구축 (Part 3에서 배운 하이브리드 검색)
vs = FAISS.from_documents(splits, embeddings)
faiss_r = vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})
bm25_r = BM25Retriever.from_documents(documents=splits, k=TOP_K)
base_retriever = EnsembleRetriever(retrievers=[faiss_r, bm25_r], weights=[0.5, 0.5])
from langchain_core.prompts import PromptTemplate
multi_query_prompt = PromptTemplate(
    input_variables=["question"],
    template="""당신은 질문 확장 전문가입니다. 사용자의 질문을 여러 관점에서 변형하여 3가지 다른 질문을 만드세요.
각 질문은 서로 다른 관점이나 표현을 사용해야 합니다. 질문만 한 줄씩 출력하고, 번호나 기호 없이 질문 내용만 출력하세요.
원본 질문: {question}"""
)


# MultiQueryRetriever 생성
multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    include_original=True,
)

print("MultiQueryRetriever 구성 완료")
print(f"  Base Retriever: EnsembleRetriever (FAISS + BM25)")
print(f"  Query 확장 LLM: gpt-4o-mini")
print(f"  include_original=True → 원본 질문 검색 결과도 포함")

MultiQueryRetriever 구성 완료
  Base Retriever: EnsembleRetriever (FAISS + BM25)
  Query 확장 LLM: gpt-4o-mini
  include_original=True → 원본 질문 검색 결과도 포함


In [21]:
demo_question = "2025년 4분기 DS 부문의 영업이익은?"

# llm_chain으로 확장 질문 생성 (원본 + 변형 3개)
expanded = multi_retriever.llm_chain.invoke({"question": demo_question})
expanded_questions = [demo_question] + list(expanded)

print("질문 확장 결과:")
print("=" * 80)
for i, q in enumerate(expanded_questions, 1):
    print(f"\n[{i}] {q}")

질문 확장 결과:

[1] 2025년 4분기 DS 부문의 영업이익은?

[2] 2025년 4분기 DS 부문에서 예상되는 영업이익은 얼마인가요?  

[3] 2025년 4분기 동안 DS 부문이 기록할 영업이익은 어떤 수준일까요?  

[4] 2025년 4분기 DS 부문의 영업이익 추정치는 어떻게 되나요?


In [22]:
demo_question = "25년 4분기 영업이익은 삼성전자와 SK하이닉스 중 누가 더 큰가?"

# llm_chain으로 확장 질문 생성 (원본 + 변형 3개)
expanded = multi_retriever.llm_chain.invoke({"question": demo_question})
expanded_questions = [demo_question] + list(expanded)

print("질문 확장 결과:")
print("=" * 80)
for i, q in enumerate(expanded_questions, 1):
    print(f"\n[{i}] {q}")

질문 확장 결과:

[1] 25년 4분기 영업이익은 삼성전자와 SK하이닉스 중 누가 더 큰가?

[2] 25년 4분기 동안 삼성전자와 SK하이닉스의 영업이익을 비교하면 어떤 결과가 나올까요?  

[3] 2025년 4분기 삼성전자와 SK하이닉스의 영업이익 중 어느 쪽이 더 높은지 알고 싶습니다.  

[4] 2025년 4분기 영업이익에서 삼성전자와 SK하이닉스의 성과를 비교해 주실 수 있나요?


## 4-4. 확장 쿼리 생성 확인

`MultiQueryRetriever`를 호출하면 내부적으로 LLM이 질문을 확장합니다.
위에서 설정한 **로깅(DEBUG)**을 통해 어떤 쿼리들이 생성되었는지 직접 확인할 수 있습니다.

In [25]:
demo_question = "25년 4분기 영업이익은 삼성전자와 SK하이닉스 중 누가 더 큰가?"

# MultiQueryRetriever invoke — 로그에 생성된 확장 쿼리가 자동 출력됨
multi_docs = multi_retriever.invoke(demo_question)
multi_pages = set(d.metadata.get('page', '?') for d in multi_docs)

print(f"\n검색된 문서 수: {len(multi_docs)}개")
print(f"검색된 페이지: {sorted(multi_pages)}")
print(f"\n상위 5개 문서 요약:\n" + "-"*40)
for i, doc in enumerate(multi_docs[:5]):
    page = doc.metadata.get('page', '?')
    snippet = doc.page_content.strip().replace('\n', ' ')
    # 한 줄에 너무 길지 않게 60자씩 끊어서 보기 좋게 출력
    max_chars = 120  # 최대 표시 글자수
    wrapped = [snippet[j:j+60] for j in range(0, min(len(snippet), max_chars), 60)]
    print(f"[{i+1}] (page {page})")
    for w in wrapped:
        print(f"    {w}")
    print("-"*40)

INFO:langchain.retrievers.multi_query:Generated queries: ['25년 4분기 동안 삼성전자와 SK하이닉스의 영업이익을 비교하면 어떤 결과가 나올까요?  ', '2025년 4분기 삼성전자와 SK하이닉스의 영업이익 중 어느 쪽이 더 높은지 알고 싶습니다.  ', '2025년 4분기 영업이익에서 삼성전자와 SK하이닉스의 성과를 비교해 주세요.']



검색된 문서 수: 13개
검색된 페이지: [4, 5, 8, 11, 12, 13, 17, 18, 20]

상위 5개 문서 요약:
----------------------------------------
[1] (page 4)
    ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 25년 4분기는 SK하이닉스
    , 26년 1분기는 삼성전자가 앞설 전망  - **25년 4분기 SK하이닉스 영업이익 더 크다**   - 2
----------------------------------------
[2] (page 17)
    ```markdown # SK하이닉스 (000660)  ## 가파르지만 편하다.  ### 25년 4분기 예상
    치 상회 SK하이닉스의 2025년 4분기 매출액은 2025년 3분기 대비 34.3% 증가한 32조 8,270
----------------------------------------
[3] (page 5)
    *출처: IBK투자증권*  ### 삼성전자, SK하이닉스 NAND B/G, ASP QoQ  #### 그림 7
    . NAND B/G QoQ 증감 비교  | 분기  | 삼성전자 | SK하이닉스 | |-------|-----
----------------------------------------
[4] (page 20)
    ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 26년 1분기 가격 상승은 
    역대급  SK하이닉스의 2026년 1분기 매출액은 25년 4분기 대비 24.7% 증가한 40.9조원으로 예상
----------------------------------------
[5] (page 12)
    ```markdown ## 삼성전자/SK하이닉스: 흔들리지 않는 편안함  ### 2026년 1분기 영업이익은
     35.3조원으로 예상  삼성전자의 2026년 1분기 영업이익은 2025년 4분기 대비 75.7% 증가한 3
-----

In [11]:
# 비교: 기본 Retriever (질문 확장 없이)
original_docs = base_retriever.invoke(demo_question)
original_pages = set(d.metadata.get('page', '?') for d in original_docs)
new_pages = multi_pages - original_pages

print(f"[기본 Retriever]   문서 {len(original_docs)}개, 페이지: {sorted(original_pages)}")
print(f"[Multi-Query]      문서 {len(multi_docs)}개, 페이지: {sorted(multi_pages)}")
print(f"\n→ Multi-Query를 통해 추가 발견된 페이지: {sorted(new_pages)} ({len(new_pages)}개)")
print(f"  질문을 다양한 관점으로 확장함으로써 기본 검색이 놓친 문서를 보완합니다.")

[기본 Retriever]   문서 7개, 페이지: [4, 5, 12, 17, 18, 20]
[Multi-Query]      문서 14개, 페이지: [2, 4, 5, 8, 11, 12, 13, 17, 18, 20]

→ Multi-Query를 통해 추가 발견된 페이지: [2, 8, 11, 13] (4개)
  질문을 다양한 관점으로 확장함으로써 기본 검색이 놓친 문서를 보완합니다.


## 4-5. RAG 답변 비교: 원본 vs Multi-Query

검색 범위가 넓어지면 LLM 답변 품질이 어떻게 달라지는지 비교합니다.

In [12]:
expected_answer = "SK하이닉스"

def run_rag(docs, retriever_name):
    context = "\n---\n".join(d.page_content for d in docs)
    t0 = time.time()
    msg = llm.invoke([HumanMessage(content=RAG_PROMPT.format(context=context, question=demo_question))])
    elapsed = time.time() - t0
    answer = msg.content if hasattr(msg, "content") else str(msg)
    contains_answer = expected_answer in answer
    print(f"\n[{retriever_name}]")
    print(f"  답변: {answer[:300]}")
    print(f"  정답 포함 여부: {'O' if contains_answer else 'X'} (정답: {expected_answer})")
    print(f"  컨텍스트 문서 수: {len(docs)}, 응답 시간: {elapsed:.2f}s")

run_rag(original_docs, "Ensemble (원본 쿼리만)")
run_rag(multi_docs, "Multi-Query (확장 쿼리 포함)")


[Ensemble (원본 쿼리만)]
  답변: 25년 4분기 영업이익은 삼성전자가 더 큽니다. 삼성전자의 영업이익은 20.1조원이고, SK하이닉스의 영업이익은 19.2조원입니다.
  정답 포함 여부: O (정답: SK하이닉스)
  컨텍스트 문서 수: 7, 응답 시간: 0.84s

[Multi-Query (확장 쿼리 포함)]
  답변: 25년 4분기 영업이익은 삼성전자가 더 큽니다. 삼성전자의 영업이익은 20.1조원이고, SK하이닉스의 영업이익은 19.2조원입니다.
  정답 포함 여부: O (정답: SK하이닉스)
  컨텍스트 문서 수: 14, 응답 시간: 0.92s


## 4-6. 노이즈 리스크와 프롬프트 설계

Multi-Query는 강력하지만 **노이즈 리스크**가 있습니다:
- 확장 질문이 원질문과 너무 동떨어지면 관련 없는 문서가 섞임
- LLM 컨텍스트에 불필요한 정보가 많아져 오히려 정확도 하락 가능

**프롬프트 설계 팁:**
- 확장 질문 수를 3~5개로 제한 (너무 많으면 노이즈 증가)
- "서로 다른 관점"을 명시하여 중복 확장 방지
- 도메인 용어를 프롬프트에 포함시켜 확장 품질 향상
- `include_original=True`로 원본 질문의 검색 결과도 반드시 포함

## 4-7. 정리

| 항목 | 설명 |
|------|------|
| **목적** | 단일 쿼리가 놓치는 관련 문서를 다양한 관점의 변형 질문으로 보완 |
| **작동** | LLM으로 N개 확장 질문 생성 → 각각 검색 → 합집합(중복 제거) |
| **장점** | 재현율(Recall) 향상, 다양한 표현/관점의 문서 커버 |
| **비용** | LLM 호출 1회 추가 (가벼운 모델 사용 가능, e.g., gpt-4o-mini) |
| **주의** | 확장 수가 너무 많거나 질이 낮으면 노이즈 유입 |

다음 Part 5에서는 검색된 문서를 **더 정밀하게 재정렬**하는 Cross-Encoder Rerank를 소개합니다.